In [1]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_community.chat_models import ChatOllama
from langchain.schema.messages import HumanMessage

# Create a ChatOllama model (e.g., using llama3, mistral, etc.)
model = ChatOllama(model="llama3.2")  # or "mistral", "gemma", etc.


# Define prompt templates (no need for separate Runnable chains)
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a comedian who tells jokes about {topic}."),
        ("human", "Tell me {joke_count} jokes."),
    ]
)

# Create the combined chain using LangChain Expression Language (LCEL)
chain = prompt_template | model | StrOutputParser()
# chain = prompt_template | model

# Run the chain
result = chain.invoke({"topic": "lawyers", "joke_count": 3})

# Output
print(result)


C:\Users\lolen\AppData\Local\Temp\ipykernel_27260\458162204.py:8: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  model = ChatOllama(model="llama3.2")  # or "mistral", "gemma", etc.


Here are three lawyer-themed jokes for you:

1. Why did the lawyer's dog go to the vet?

Because it was feeling a little ruff on its contract! (get it?)

2. What do you call a lawyer who doesn't charge by the hour?

A partner in crime. (ba-dum-tss)

3. Why did the lawyer take his client to the amusement park?

To show him that they could have a 'fun' time defending their case... just kidding, it was actually to prove that even in a thrill ride of a lawsuit, they were always in control!


In [4]:
from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableLambda, RunnableSequence
from langchain_community.chat_models import ChatOllama
from langchain.schema.messages import HumanMessage

# Create a ChatOllama model (e.g., using llama3, mistral, etc.)
model = ChatOllama(model="llama3.2")  # or "mistral", "gemma", etc.

# Define prompt templates
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a comedian who tells jokes about {topic}."),
        ("human", "Tell me {joke_count} jokes."),
    ]
)

# Create individual runnables (steps in the chain)
format_prompt = RunnableLambda(lambda x: prompt_template.format_prompt(**x))
invoke_model = RunnableLambda(lambda x: model.invoke(x.to_messages()))
parse_output = RunnableLambda(lambda x: x.content)

# Create the RunnableSequence (equivalent to the LCEL chain)
chain = RunnableSequence(first=format_prompt, middle=[invoke_model], last=parse_output)

# Run the chain
response = chain.invoke({"topic": "engineers", "joke_count": 3})

# Output
print(response)


Here are three engineer-themed jokes for you:

1. Why did the engineer cross the road? To get to the other side... of the code review process! (get it?)

2. Why do engineers make great partners? Because they're always willing to take a "load" off your shoulders, and also because they can calculate the trajectory of your romantic relationship.

3. What did the structural engineer say to the building that was having trouble standing? Don't worry, I've got a solid foundation... for a good argument!


In [2]:


# Define prompt templates
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a comedian who tells jokes about {topic}."),
        ("human", "Tell me {joke_count} jokes."),
    ]
)

# Define additional processing steps using RunnableLambda
uppercase_output = RunnableLambda(lambda x: x.upper())
count_words = RunnableLambda(lambda x: f"Word count: {len(x.split())}\n{x}")

# Create the combined chain using LangChain Expression Language (LCEL)
chain = prompt_template | model | StrOutputParser() | uppercase_output | count_words

# Run the chain
result = chain.invoke({"topic": "doctors", "joke_count": 3})

# Output
print(result)


NameError: name 'RunnableLambda' is not defined

In [3]:

from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableParallel, RunnableLambda

# Define prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert product reviewer."),
        ("human", "List the main features of the product {product_name}."),
    ]
)


# Define pros analysis step
def analyze_pros(features):
    pros_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert product reviewer."),
            (
                "human",
                "Given these features: {features}, list the pros of these features.",
            ),
        ]
    )
    return pros_template.format_prompt(features=features)


# Define cons analysis step
def analyze_cons(features):
    cons_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert product reviewer."),
            (
                "human",
                "Given these features: {features}, list the cons of these features.",
            ),
        ]
    )
    return cons_template.format_prompt(features=features)


# Combine pros and cons into a final review
def combine_pros_cons(pros, cons):
    return f"Pros:\n{pros}\n\nCons:\n{cons}"


# Simplify branches with LCEL
pros_branch_chain = (
    RunnableLambda(lambda x: analyze_pros(x)) | model | StrOutputParser()
)

cons_branch_chain = (
    RunnableLambda(lambda x: analyze_cons(x)) | model | StrOutputParser()
)

# Create the combined chain using LangChain Expression Language (LCEL)
chain = (
    prompt_template
    | model
    | StrOutputParser()
    | RunnableParallel(branches={"pros": pros_branch_chain, "cons": cons_branch_chain})
    | RunnableLambda(lambda x: combine_pros_cons(x["branches"]["pros"], x["branches"]["cons"]))
)

# Run the chain
result = chain.invoke({"product_name": "Nike Air Max 270"})

# Output
print(result)


Pros:
Here are the pros of each feature of the Nike Air Max 270:

**1. Full-Length Air Unit:**
- Provides maximum cushioning and shock absorption in the midsole
- Offers improved comfort and reduced impact on joints during running
- Helps to increase overall performance and reduce fatigue

**2. Max Air Technology:**
- Provides a more responsive and springy feel with each step
- Enhances the shoe's ability to absorb shock and provide comfort
- Adds to the overall sense of comfort and responsiveness of the shoe

**3. Flyknit Upper:**
- Offers a snug and comfortable fit due to its lightweight and flexible material
- Provides improved breathability and moisture management
- Helps to reduce the risk of blisters and discomfort during wear

**4. Durable Outsole:**
- Provides traction and grip on various surfaces, reducing the risk of slipping or falling
- Can withstand heavy use and maintain performance over time
- Offers a long-lasting and reliable ride for runners and athletes

**5. Self-La

In [ ]:

from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableBranch
from langchain_openai import ChatOpenAI


# Define prompt templates for different feedback types
positive_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human",
         "Generate a thank you note for this positive feedback: {feedback}."),
    ]
)

negative_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human",
         "Generate a response addressing this negative feedback: {feedback}."),
    ]
)

neutral_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        (
            "human",
            "Generate a request for more details for this neutral feedback: {feedback}.",
        ),
    ]
)

escalate_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        (
            "human",
            "Generate a message to escalate this feedback to a human agent: {feedback}.",
        ),
    ]
)

# Define the feedback classification template
classification_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human",
         "Classify the sentiment of this feedback as positive, negative, neutral, or escalate: {feedback}."),
    ]
)

# Define the runnable branches for handling feedback
branches = RunnableBranch(
    (
        lambda x: "positive" in x,
        positive_feedback_template | model | StrOutputParser()  # Positive feedback chain
    ),
    (
        lambda x: "negative" in x,
        negative_feedback_template | model | StrOutputParser()  # Negative feedback chain
    ),
    (
        lambda x: "neutral" in x,
        neutral_feedback_template | model | StrOutputParser()  # Neutral feedback chain
    ),
    escalate_feedback_template | model | StrOutputParser()
)

# Create the classification chain
classification_chain = classification_template | model | StrOutputParser()

# Combine classification and response generation into one chain
chain = classification_chain | branches

# Run the chain with an example review
# Good review - "The product is excellent. I really enjoyed using it and found it very helpful."
# Bad review - "The product is terrible. It broke after just one use and the xquality is very poor."
# Neutral review - "The product is okay. It works as expected but nothing exceptional."
# Default - "I'm not sure about the product yet. Can you tell me more about its features and benefits?"

review = "The product is terrible. It broke after just one use and the quality is very poor."
result = chain.invoke({"feedback": review})

# Output the result
print(result)


Thank you for taking the time to share your thoughts on my performance. I'm sorry to hear that you have a negative impression of our interaction.

I'd like to understand more about what specifically led you to feel this way. Could you please elaborate on what didn't meet your expectations? Was there something I said or did that caused you concern, and if so, what was it?

By gathering more information, I can better comprehend the issue at hand and work towards a resolution or improvement. Your feedback is invaluable in helping me grow and provide a better experience for others in the future.

Please know that I'm here to listen and learn from your perspective.
